# Ekstraksi Fitur Data kualitas udara di wilayah Dukun, Gresik


## Ekstraksi 68 Fitur TSFEL untuk Polutan

In [1]:
!pip install tsfel

In [2]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
df = pd.read_csv('3Polutan_Dukun_Final.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

# --- FIX: paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN ---
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")

Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- 2. Daftar PERSIS fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")
extracted_features_final.to_csv(f'{target_pollutant}_3Dukun_TSFEL.csv', index=False)

Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: 52
Jumlah fitur yang diminta: 68


Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68


In [3]:
import pandas as pd

df_fitur = pd.read_csv("NO2_3Dukun_TSFEL.csv")
pd.set_option('display.max_columns', None)
display(df_fitur)

,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,calc_var,dfa,distance,ecdf,ecdf_percentile,ecdf_percentile_count,ecdf_slope,entropy,fundamental_frequency,higuchi_fractal_dimension,hist_mode,human_range_energy,hurst_exponent,interq_range,kurtosis,lempel_ziv,lpcc,max_frequency,max_power_spectrum,maximum_fractal_length,mean_abs_deviation,mean_abs_diff,mean_diff,median_abs_deviation,median_abs_diff,median_diff,median_frequency,mfcc,mse,negative_turning,neighbourhood_peaks,petrosian_fractal_dimension,pk_pk_distance,positive_turning,power_bandwidth,rms,skewness,slope,spectral_centroid,spectral_decrease,spectral_distance,spectral_entropy,spectral_kurtosis,spectral_positive_turning,spectral_roll_off,spectral_roll_on,spectral_skewness,spectral_slope,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,3.581219e-07,0.01069,3.0,9.838513e-10,202.855759,0.000063,0.000029,0.000028,0.000003,0.000011,1.189720e-10,0.899556,364.0,0.015068,0.000029,182.5,31125.376776,0.999356,0.00274,1.904593,0.000024,0.0,0.821334,0.000015,-0.063519,0.180822,0.670898,0.421918,70.722928,-2.53983,0.000009,0.000006,-6.630420e-08,0.000007,0.000004,-4.189349e-07,0.068493,13.820322,1.236576,55.0,15.0,1.019843,0.00006,55.0,0.276712,0.000031,0.550903,2.774785e-08,0.123726,-2.235568,-1.73083,0.712419,2.992151,57.0,0.421918,0.0,1.034444,-0.032763,0.139943,0.344302,1.612152e-10,0.002004,0.000002,0.000017,2.150886,0.000017,3.160192e-10,0.0


In [4]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
df = pd.read_csv('3Polutan_Dukun_Final.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'CO'

# --- FIX: paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN ---
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")

Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- 2. Daftar PERSIS fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")
extracted_features_final.to_csv(f'{target_pollutant}_3Dukun_TSFEL.csv', index=False)

Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: 47
Jumlah fitur yang diminta: 68


Berhasil! Jumlah fitur yang dihasilkan untuk CO: 68


In [5]:
import pandas as pd

df_fitur = pd.read_csv("CO_3Dukun_TSFEL.csv")
pd.set_option('display.max_columns', None)
display(df_fitur)

,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,calc_var,dfa,distance,ecdf,ecdf_percentile,ecdf_percentile_count,ecdf_slope,entropy,fundamental_frequency,higuchi_fractal_dimension,hist_mode,human_range_energy,hurst_exponent,interq_range,kurtosis,lempel_ziv,lpcc,max_frequency,max_power_spectrum,maximum_fractal_length,mean_abs_deviation,mean_abs_diff,mean_diff,median_abs_deviation,median_abs_diff,median_diff,median_frequency,mfcc,mse,negative_turning,neighbourhood_peaks,petrosian_fractal_dimension,pk_pk_distance,positive_turning,power_bandwidth,rms,skewness,slope,spectral_centroid,spectral_decrease,spectral_distance,spectral_entropy,spectral_kurtosis,spectral_positive_turning,spectral_roll_off,spectral_roll_on,spectral_skewness,spectral_slope,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,0.31295,10.58842,3.0,0.00086,182.402254,0.038594,0.029091,0.028833,0.020323,0.003336,0.000011,0.917194,364.00145,0.015068,0.028931,182.5,99.398302,0.999356,0.00274,1.852419,0.028545,0.0,0.802562,0.004875,-0.174817,0.175342,0.906714,0.421918,21.052426,-0.092724,0.002711,0.0019,0.000013,0.002421,0.001122,-0.000189,0.0,17.497902,1.238981,72.0,16.0,1.025637,0.018271,72.0,0.416438,0.029281,0.304184,5.668754e-07,0.081209,-7.002415,-1192.320387,0.777051,4.49369,57.0,0.421918,0.0,1.64552,-0.043854,0.133953,0.534442,0.000017,0.691652,0.00178,0.007723,2.12092,0.007492,0.000064,0.0


In [6]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
df = pd.read_csv('3Polutan_Dukun_Final.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'SO2'

# --- FIX: paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN ---
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")

Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- 2. Daftar PERSIS fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")
extracted_features_final.to_csv(f'{target_pollutant}_3Dukun_TSFEL.csv', index=False)

Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: 54
Jumlah fitur yang diminta: 68


Berhasil! Jumlah fitur yang dihasilkan untuk SO2: 68


In [7]:
import pandas as pd

df_fitur = pd.read_csv("SO2_3Dukun_TSFEL.csv")
pd.set_option('display.max_columns', None)
display(df_fitur)

,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,calc_var,dfa,distance,ecdf,ecdf_percentile,ecdf_percentile_count,ecdf_slope,entropy,fundamental_frequency,higuchi_fractal_dimension,hist_mode,human_range_energy,hurst_exponent,interq_range,kurtosis,lempel_ziv,lpcc,max_frequency,max_power_spectrum,maximum_fractal_length,mean_abs_deviation,mean_abs_diff,mean_diff,median_abs_deviation,median_abs_diff,median_diff,median_frequency,mfcc,mse,negative_turning,neighbourhood_peaks,petrosian_fractal_dimension,pk_pk_distance,positive_turning,power_bandwidth,rms,skewness,slope,spectral_centroid,spectral_decrease,spectral_distance,spectral_entropy,spectral_kurtosis,spectral_positive_turning,spectral_roll_off,spectral_roll_on,spectral_skewness,spectral_slope,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,0.000008,0.038811,2.0,2.320691e-08,206.206722,0.000419,0.000042,0.000045,-0.000333,0.000146,2.141007e-08,0.823196,364.000004,0.015068,0.000035,182.5,3123.502867,0.999356,0.00274,1.922317,0.000081,0.0,0.761082,0.000177,0.081365,0.19726,0.225456,0.452055,22.360043,-1.338295,0.000113,0.000099,1.878333e-07,0.000089,0.000078,0.00001,0.161644,32.372619,1.219648,70.0,15.0,1.025131,0.000752,71.0,0.378082,0.000152,0.021089,3.176759e-07,0.183897,-0.150666,-10.209664,0.834021,2.133064,62.0,0.452055,0.00274,0.509109,-0.017066,0.142119,0.26992,3.613930e-08,0.036125,0.000003,0.000206,2.167157,0.000206,4.447452e-08,94.0


# Penjelasan 68 Fitur TSFEL untuk Data NO2

Dokumen ini menjelaskan arti dan kegunaan dari 68 fitur yang diekstraksi menggunakan pustaka **TSFEL** dari data time series konsentrasi NO2 harian. Fitur dikelompokkan menjadi 4 domain: **Statistik**, **Temporal**, **Spektral**, dan **Wavelet**.

---

## A. Domain Statistik
*Menggambarkan sebaran dan karakteristik nilai NO2 secara keseluruhan, tanpa memperhatikan urutan waktu.*

- **`calc_mean`** — Nilai rata-rata NO2 selama periode data. Kegunaan: menunjukkan level polusi rata-rata suatu kecamatan.
    $$\bar{x} = \frac{1}{N}\sum_{i=1}^{N} x_i$$
- **`calc_median`** — Nilai tengah data setelah diurutkan. Kegunaan: representasi pusat data yang tidak mudah terpengaruh nilai ekstrem, cocok jika distribusinya condong (skewed).
$$
\text{median}(x)=
\begin{cases}
x_{\left(\frac{N+1}{2}\right)} & N \text{ ganjil}\\[4pt]
\dfrac{1}{2}\left(x_{\left(\frac{N}{2}\right)}+x_{\left(\frac{N}{2}+1\right)}\right) & N \text{ genap}
\end{cases}
$$
- **`calc_std`** — Standar deviasi, mengukur seberapa besar fluktuasi nilai dari rata-rata. Kegunaan: kecamatan dengan std tinggi berarti kualitas udaranya lebih fluktuatif/tidak stabil.
$$\sigma = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(x_i-\bar{x})^2}$$
- **`calc_var`** — Variansi (kuadrat dari std). Kegunaan: ukuran sebaran data secara keseluruhan, versi lain dari std.
$$\sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i-\bar{x})^2$$
- **`calc_max`** — Nilai NO2 tertinggi yang tercatat. Kegunaan: mengetahui puncak polusi selama periode data.
$$x_{max}=\max_{i}(x_i) \qquad x_{min}=\min_{i}(x_i)$$
- **`calc_min`** — Nilai NO2 terendah yang tercatat. Kegunaan: mengetahui kondisi udara terbaik selama periode data.
- **`skewness`** — Kemencengan distribusi (apakah data condong ke nilai tinggi atau rendah). Kegunaan: mendeteksi apakah lonjakan polusi terjadi sesekali (ekor panjang) atau merata.
$$\text{skew} = \frac{\dfrac{1}{N}\sum_{i=1}^{N}(x_i-\bar{x})^3}{\sigma^3}$$
- **`kurtosis`** — Keruncingan distribusi dibanding distribusi normal. Kegunaan: mendeteksi seberapa sering muncul nilai ekstrem/lonjakan tajam.
$$\text{kurt} = \frac{\dfrac{1}{N}\sum_{i=1}^{N}(x_i-\bar{x})^4}{\sigma^4} - 3$$
- **`interq_range`** — Rentang interkuartil (Q3 − Q1), sebaran 50% data tengah. Kegunaan: ukuran variabilitas yang tahan terhadap outlier.
- **`mean_abs_deviation`** — Rata-rata deviasi absolut terhadap mean. Kegunaan: ukuran variabilitas alternatif, kurang sensitif terhadap outlier ekstrem dibanding std.
$$MAD = \frac{1}{N}\sum_{i=1}^{N}\left|x_i-\bar{x}\right|$$
- **`median_abs_deviation`** — Median dari deviasi absolut terhadap median. Kegunaan: ukuran sebaran yang sangat robust terhadap outlier, cocok untuk data polusi yang kadang melonjak tajam.
$$MedAD = \text{median}\left(\left|x_i-\text{median}(x)\right|\right)$$
- **`ecdf`** — *Empirical Cumulative Distribution Function*, kurva yang menunjukkan proporsi data di bawah suatu nilai. Kegunaan: memberi gambaran ringkas distribusi kumulatif nilai NO2.
$$F(x_i)=\frac{i}{N}, \qquad \overline{F}=\frac{1}{N}\sum_{i=1}^{N}F(x_i)$$
- **`ecdf_percentile`** — Nilai NO2 pada persentil-persentil tertentu (mis. 25%, 75%). Kegunaan: melihat sebaran nilai pada titik persentil penting.
$$x_{(p)} \;\; \text{sedemikian sehingga} \;\; F\big(x_{(p)}\big)=p$$
- **`ecdf_percentile_count`** — Jumlah titik data di bawah persentil acuan. ($x_{(p)}$) Kegunaan: melengkapi `ecdf_percentile` dengan info kuantitas datanya.
$$n_{(p)} = \#\{\,x_i \le x_{(p)}\,\}$$
- **`ecdf_slope`** — Kemiringan kurva ECDF antara dua persentil. Kegunaan: menunjukkan seberapa cepat nilai data terkonsentrasi pada suatu rentang.
$$\text{slope} = \frac{x_{(p_{end})}-x_{(p_{init})}}{p_{end}-p_{init}}$$
- **`hist_mode`** — Nilai yang paling sering muncul (modus) berdasarkan histogram. Kegunaan: mengetahui level NO2 "tipikal" yang paling sering terjadi.
$$\text{mode} = \underset{b}{\arg\max}\;h(b)$$
- **`pk_pk_distance`** — *Peak-to-peak distance*, selisih nilai maksimum dan minimum. Kegunaan: mengukur rentang total fluktuasi NO2 selama periode data.
$$PP = x_{max}-x_{min}$$
- **`rms`** — *Root Mean Square*, akar dari rata-rata kuadrat nilai. Kegunaan: mengukur "besaran energi" rata-rata sinyal, memperhitungkan magnitude keseluruhan (bukan hanya rata-rata biasa).
$$RMS = \sqrt{\frac{1}{N}\sum_{i=1}^{N}x_i^{2}}$$

---

## B. Domain Temporal
*Menggambarkan pola perubahan nilai NO2 dari hari ke hari, termasuk tren, osilasi, dan kompleksitas sinyal.*

- **`autocorr`** — Autokorelasi, mengukur kemiripan nilai hari ini dengan nilai di waktu sebelumnya. Kegunaan: mendeteksi pola berulang/musiman pada data.
$$R(\tau) = \sum_{i=1}^{N-\tau} x_i\,x_{i+\tau}$$
- **`distance`** — Total panjang lintasan kurva sinyal (jumlah jarak antar titik berurutan). Kegunaan: mengukur seberapa "aktif"/berliku pergerakan NO2 dari hari ke hari.
$$D = \sum_{i=1}^{N-1}\sqrt{1+(x_{i+1}-x_i)^2}$$
- **`slope`** — Kemiringan garis tren linear yang di-fit ke seluruh data. Kegunaan: menunjukkan apakah NO2 cenderung naik atau turun secara keseluruhan.
$$a=\frac{N\sum_i t_i x_i - \sum_i t_i \sum_i x_i}{N\sum_i t_i^{2}-\left(\sum_i t_i\right)^2}$$
- **`entropy`** — Mengukur tingkat ketidakteraturan/ketidakpastian data. Kegunaan: nilai tinggi berarti pola NO2 lebih acak dan sulit diprediksi.
$$H = -\sum_{i} p_i \log_2 p_i$$
- **`zero_cross`** — Jumlah kali sinyal melewati garis nol/rata-rata. Kegunaan: mengukur seberapa sering nilai berosilasi naik-turun melewati titik tengahnya.
$$ZC=\sum_{i=1}^{N-1}\mathbb{1}\Big[\text{sign}(x_i-\bar{x})\neq\text{sign}(x_{i+1}-\bar{x})\Big]$$
- **`mean_diff`** — Rata-rata selisih antar titik berurutan. Kegunaan: menunjukkan kecenderungan rata-rata perubahan harian (naik/turun).
$$\overline{\Delta x}=\frac{1}{N-1}\sum_{i=1}^{N-1}(x_{i+1}-x_i)$$
- **`median_diff`** — Median dari selisih antar titik berurutan. Kegunaan: versi robust dari `mean_diff`, tidak terlalu dipengaruhi lonjakan ekstrem.
$$\text{median}(x_{i+1}-x_i)$$
- **`mean_abs_diff`** — Rata-rata nilai absolut selisih antar titik. Kegunaan: mengukur besar rata-rata perubahan harian tanpa memandang arah naik/turun.
$$\frac{1}{N-1}\sum_{i=1}^{N-1}\left|x_{i+1}-x_i\right|$$
- **`median_abs_diff`** — Median nilai absolut selisih antar titik. Kegunaan: versi robust dari `mean_abs_diff`.
$$\text{median}\left(\left|x_{i+1}-x_i\right|\right)$$
- **`sum_abs_diff`** — Total akumulasi seluruh perubahan absolut sepanjang periode. Kegunaan: mengukur total "aktivitas"/volatilitas keseluruhan sinyal.
$$\sum_{i=1}^{N-1}\left|x_{i+1}-x_i\right|$$
- **`positive_turning`** — Jumlah titik balik naik (dari turun ke naik). Kegunaan: menghitung frekuensi terjadinya kenaikan mendadak/local minimum.
$$PT=\sum_{i=2}^{N-1}\mathbb{1}\Big[(x_i-x_{i-1})<0 \;\wedge\; (x_{i+1}-x_i)>0\Big]$$
- **`negative_turning`** — Jumlah titik balik turun (dari naik ke turun). Kegunaan: menghitung frekuensi puncak/local maximum polusi singkat.
$$NT=\sum_{i=2}^{N-1}\mathbb{1}\Big[(x_i-x_{i-1})>0 \;\wedge\; (x_{i+1}-x_i)<0\Big]$$
- **`neighbourhood_peaks`** — Jumlah puncak (peak) lokal yang terdeteksi dalam sinyal. Kegunaan: mengidentifikasi seberapa sering muncul lonjakan konsentrasi NO2.
$$\text{peak}_i=\mathbb{1}\Big[x_i>x_j,\ \forall j\in[i-n,\,i+n],\ j\neq i\Big]$$
- **`calc_centroid`** — Titik pusat massa sinyal berdasarkan waktu. Kegunaan: menunjukkan periode di mana "bobot" konsentrasi NO2 secara rata-rata terkonsentrasi.
$$C=\frac{\sum_i t_i x_i}{\sum_i x_i}$$
- **`auc`** — *Area Under Curve*, luas area di bawah kurva sinyal. Kegunaan: mengukur total "paparan" polusi kumulatif selama periode data.
$$AUC \approx \sum_{i=1}^{N-1}\frac{x_i+x_{i+1}}{2}\,\Delta t$$
- **`abs_energy`** — Jumlah kuadrat seluruh nilai sinyal. Kegunaan: mengukur total intensitas/magnitude sinyal secara keseluruhan.
$$E=\sum_{i=1}^{N}x_i^{2}$$
- **`mse`** — *Multiscale entropy*, mengukur kompleksitas sinyal pada berbagai skala waktu. Kegunaan: menilai apakah pola NO2 kompleks/multi-skala atau sederhana.
$$y_j^{(\tau)}=\frac{1}{\tau}\sum_{i=(j-1)\tau+1}^{j\tau}x_i \;\Rightarrow\; \overline{MSE}=\frac{1}{T}\sum_{\tau=1}^{T} SampEn\big(y^{(\tau)}\big)$$
- **`dfa`** — *Detrended Fluctuation Analysis*, mengukur korelasi jangka panjang dalam sinyal setelah tren dihilangkan. Kegunaan: mendeteksi apakah polusi hari ini berhubungan dengan pola beberapa waktu sebelumnya secara jangka panjang.
$$y(k)=\sum_{i=1}^{k}(x_i-\bar{x}), \qquad F(n)=\sqrt{\frac{1}{N}\sum_{k=1}^{N}\big(y(k)-y_n(k)\big)^2}, \qquad F(n)\propto n^{\alpha}$$
($\alpha$ = slope regresi $\log F(n)$ vs $\log n$, itulah nilai fitur `dfa`)
- **`hurst_exponent`** — Mengukur kecenderungan sinyal untuk terus trending (*persistent*) atau berbalik arah menuju rata-rata (*mean-reverting*). Kegunaan: menilai apakah tren NO2 cenderung berlanjut atau kembali normal.
$$\frac{R(n)}{S(n)} \propto n^{H}$$
($R(n)$ = rentang deviasi kumulatif, $S(n)$ = std pada segmen ukuran $n$; $H$ = slope regresi $\log(R/S)$ vs $\log n$)
- **`higuchi_fractal_dimension`** — Dimensi fraktal (metode Higuchi), mengukur kompleksitas/kekasaran bentuk kurva. Kegunaan: nilai tinggi menunjukkan sinyal lebih "bergerigi"/tidak teratur.
$$L_m(k)=\frac{N-1}{\left\lfloor\frac{N-m}{k}\right\rfloor k^2}\sum_{i=1}^{\lfloor (N-m)/k\rfloor}\left|x_{m+ik}-x_{m+(i-1)k}\right|, \qquad L(k)\propto k^{-D}$$
($D$ = slope regresi $\log L(k)$ vs $\log(1/k)$)
- **`petrosian_fractal_dimension`** — Metode alternatif menghitung dimensi fraktal berdasarkan perubahan arah sinyal. Kegunaan: mengukur kompleksitas pola dengan pendekatan komputasi lebih ringan dibanding Higuchi.
$$PFD=\frac{\log_{10}N}{\log_{10}N+\log_{10}\!\left(\dfrac{N}{N+0.4\,N_\delta}\right)}$$
($N_\delta$ = jumlah perubahan tanda pada selisih sinyal)
- **`maximum_fractal_length`** — Panjang maksimum kurva pada skala fraktal tertentu. Kegunaan: melengkapi info fraktal soal seberapa panjang lintasan sinyal pada skala terkecil.
$$L(1)=\frac{1}{N-1}\sum_{i=1}^{N-1}\left|x_{i+1}-x_i\right|$$
- **`lempel_ziv`** — Kompleksitas Lempel-Ziv, mengukur "keteracakan"/kompresibilitas pola sinyal. Kegunaan: nilai tinggi berarti pola NO2 sulit diprediksi/dikompresi, menandakan variasi yang kompleks.
$$LZC=\frac{c(n)}{n/\log_2 n}$$
($c(n)$ = jumlah pola/substring unik yang ditemukan saat sinyal (dibinerisasi terhadap median) di-parsing dari awal ke akhir)

---

## C. Domain Spektral
*Menggambarkan pola frekuensi/periodisitas sinyal, diperoleh dari transformasi Fourier terhadap data NO2.*

- **`spectral_centroid`** — Titik pusat massa spektrum frekuensi. Kegunaan: mengindikasikan seberapa cepat/lambat pola perubahan NO2 berosilasi secara dominan.
$$C_f=\frac{\sum_i f_i P(f_i)}{\sum_i P(f_i)}$$
- **`spectral_spread`** — Sebaran (variance) di sekitar spectral centroid. Kegunaan: mengukur seberapa lebar rentang frekuensi yang mendominasi pola sinyal.
$$S_f=\sqrt{\frac{\sum_i P(f_i)(f_i-C_f)^2}{\sum_i P(f_i)}}$$
- **`spectral_skewness`** — Kemencengan distribusi spektrum frekuensi. Kegunaan: menunjukkan apakah energi sinyal condong ke frekuensi rendah (tren jangka panjang) atau tinggi (fluktuasi cepat).
$$\frac{\sum_i P(f_i)(f_i-C_f)^3}{S_f^{3}\sum_i P(f_i)}$$
- **`spectral_kurtosis`** — Keruncingan distribusi spektrum. Kegunaan: mendeteksi apakah ada satu frekuensi dominan yang sangat menonjol.
$$\frac{\sum_i P(f_i)(f_i-C_f)^4}{S_f^{4}\sum_i P(f_i)}$$
- **`spectral_entropy`** — Entropi dari distribusi energi pada spektrum frekuensi. Kegunaan: mengukur seberapa "tersebar" energi sinyal di berbagai frekuensi (makin tinggi makin acak/noise-like).
$p_i = P(f_i)/\sum_i P(f_i)$
$$H_f=-\sum_i p_i \log_2 p_i$$
- **`spectral_slope`** — Kemiringan tren penurunan/kenaikan energi seiring bertambahnya frekuensi. Kemiringan regresi linear $P(f_i)\approx a f_i + b$ (ambil $a$). Kegunaan: menunjukkan dominasi komponen frekuensi rendah vs tinggi. 
- **`spectral_decrease`** — Mengukur seberapa cepat energi menurun terhadap frekuensi. Kegunaan: melihat konsentrasi energi pada frekuensi rendah (mirip `spectral_slope` dengan pembobotan berbeda).
$$SD=\frac{1}{\sum_{i=2}^{M}P(f_i)}\sum_{i=2}^{M}\frac{P(f_i)-P(f_1)}{i-1}$$
- **`spectral_distance`** — Jarak kumulatif antara kurva spektrum aktual dan garis linear ideal. Kegunaan: mengukur seberapa jauh bentuk spektrum menyimpang dari pola sederhana.
$$SDist=\sum_i \Big|\text{cumsum}\big(P(f)\big)_i - \text{line}_i\Big|$$
- **`spectral_variation`** — Perubahan/variasi spektrum antar segmen sinyal. Kegunaan: mendeteksi seberapa stabil pola frekuensi sinyal dari waktu ke waktu.
$$SV = 1-\frac{\sum_i P_t(f_i)\,P_{t-1}(f_i)}{\sqrt{\sum_i P_t(f_i)^2}\sqrt{\sum_i P_{t-1}(f_i)^2}}$$
- **`spectral_positive_turning`** — Jumlah titik balik naik pada representasi spektral sinyal. Rumus sama seperti `positive_turning`, diterapkan pada deret $P(f_i)$. Kegunaan: melihat variasi/osilasi pada domain frekuensi.
- **`spectral_roll_off`** — Frekuensi di bawah mana sebagian besar energi sinyal terkumpul (mis. 95%). Kegunaan: menunjukkan batas atas frekuensi yang dominan.
$$\sum_{f_i \le f_{95}} P(f_i) = 0.95\sum_i P(f_i)$$
- **`spectral_roll_on`** — Frekuensi di mana energi awal sinyal mulai terkumpul. Kegunaan: menandai batas bawah rentang frekuensi signifikan.
$$\sum_{f_i \le f_{5}} P(f_i) = 0.05\sum_i P(f_i)$$
- **`max_frequency`** — Frekuensi dengan magnitude tertinggi pada spektrum. Kegunaan: menunjukkan periodisitas paling dominan (mis. siklus mingguan/bulanan).
- **`median_frequency`** — Frekuensi di titik tengah distribusi energi kumulatif. Kegunaan: representasi frekuensi "khas" dari sinyal.
- **`fundamental_frequency`** — Frekuensi dasar/komponen periodik terkuat pada sinyal. Kegunaan: mengindikasikan siklus utama, misalnya pola musiman NO2.
$$f_0=\underset{f_i\,>\,0}{\arg\max}\;P(f_i)$$
(puncak magnitude pertama pada spektrum, di luar komponen DC)
- **`max_power_spectrum`** — Nilai daya (power) maksimum pada spektrum frekuensi. Kegunaan: menunjukkan seberapa kuat komponen periodik paling dominan.
- **`power_bandwidth`** — Lebar pita frekuensi yang mengandung sebagian besar daya sinyal. Kegunaan: mengukur seberapa lebar rentang frekuensi yang berkontribusi signifikan.
- **`average_power`** — Rata-rata daya sinyal. Kegunaan: representasi umum "kekuatan" sinyal secara keseluruhan.
- **`human_range_energy`** — Proporsi energi sinyal pada rentang frekuensi acuan tertentu. Kegunaan: memberi info energi relatif sinyal pada skala waktu/frekuensi spesifik.
- **`mfcc`** — *Mel-Frequency Cepstral Coefficients*, koefisien yang merangkum bentuk spektrum dalam skala mel (umum dipakai di pengolahan audio). Kegunaan: menangkap pola bentuk spektral secara ringkas, berguna sebagai fitur pembeda pola antar kecamatan.
- **`lpcc`** — *Linear Prediction Cepstral Coefficients*, koefisien dari model prediksi linear sinyal. Kegunaan: merangkum karakteristik "bentuk" sinyal berdasarkan model autoregresif, melengkapi info spektral lainnya.
- **`spectrogram_mean_coeff`** — Rata-rata koefisien spectrogram (representasi energi sinyal terhadap waktu & frekuensi). Kegunaan: memberi gambaran umum bagaimana energi sinyal berubah lintas waktu-frekuensi.

---

## D. Domain Wavelet
*Dekomposisi sinyal ke berbagai skala waktu-frekuensi, menangkap pola jangka pendek dan panjang sekaligus.*

- **`wavelet_abs_mean`** — Rata-rata nilai absolut koefisien wavelet pada berbagai skala dekomposisi. Kegunaan: mengukur besaran rata-rata komponen sinyal di skala waktu-frekuensi berbeda.
- **`wavelet_energy`** — Total energi pada koefisien wavelet. Kegunaan: menunjukkan seberapa besar "kekuatan" sinyal tersebar di berbagai skala.
- **`wavelet_entropy`** — Entropi dari distribusi energi antar skala wavelet. Kegunaan: mengukur seberapa merata/acak distribusi energi sinyal antar skala waktu.
- **`wavelet_std`** — Standar deviasi koefisien wavelet. Kegunaan: mengukur variabilitas komponen sinyal pada tiap skala.
- **`wavelet_var`** — Variansi koefisien wavelet. Kegunaan: serupa `wavelet_std` namun dalam skala kuadrat, menekankan komponen dengan variasi besar.

---

## Ringkasan

| Domain | Jumlah Fitur | Fokus Analisis |
|---|---|---|
| Statistik | 18 | Sebaran nilai NO2 secara keseluruhan |
| Temporal | 23 | Pola perubahan nilai antar waktu & kompleksitas sinyal |
| Spektral | 22 | Pola frekuensi/periodisitas sinyal |
| Wavelet | 5 | Pola pada berbagai skala waktu-frekuensi sekaligus |
| **Total** | **68** | |

## Simpan Hasil Ekstraksi ke Database Aiven

Contoh ini menggunakan **PostgreSQL** (jenis service Aiven yang paling umum dipakai).

Sebelum menyambungkan koneksi melalui aplikasi apa pun, kita membutuhkan informasi kredensial server.
1. Akses *dashboard* atau console **Aiven**, lalu arahkan ke proyek yang dimiliki.
2. Buka tab **Overview** pada layanan (*service*) PostgreSQL yang sedang beroperasi (`pg-157c2e9`).
3. Pada bagian **Connection information**, catat parameter-parameter berikut ini:
   * **Host:** `pg-157c2e9-project-95fb.l.aivencloud.com`
   * **Port:** `13459`
   * **User:** `avnadmin`
   * **Password:** `AVNS_Khi-ZHOCTI3NhlU_Dui`
   * **SSL mode:** `require`
   * **Database:** `polutan_dukun_tsfel`

![Aiven PostgreSQL Console](../img/aiven2.png)


### Data di Database